In [1]:
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments
from trl import RewardTrainer, RewardConfig

# --- 1. Конфигурация ---
SFT_MODEL_ID = "HuggingFaceTB/SmolLM2-135M-Instruct"
RM_SAVE_PATH = "./reward_model_smollm2"
MAX_LENGTH = 2048 # Ограничиваем длину, чтобы избежать OOM

# --- 2. Загрузка датасета и его подготовка ---
print("Загрузка датасета HelpSteer2-binarized...")
dataset = load_dataset("juyoungml/HelpSteer2-binarized", split="train")

# Разделяем на train и validation
dataset = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = dataset["train"]
eval_dataset = dataset["test"]

print(f"Размер трейн-сплита: {len(train_dataset)}")
print(f"Размер eval-сплита: {len(eval_dataset)}")
print("Пример данных:", train_dataset[0])

Загрузка датасета HelpSteer2-binarized...
Размер трейн-сплита: 6501
Размер eval-сплита: 723
Пример данных: {'prompt': 'In navigating the competitive landscape of their industry, how can a small company owner strategically enhance client retention and loyalty? Provide a detailed action plan outlining specific operations, considering constraints such as a limited budget for implementation. Additionally, discuss the role of technology in fostering client loyalty and how industry-specific challenges impact the proposed strategies. Offer insights into the integration of customer feedback and experiences, and provide data-driven case studies of successful client retention efforts in similar business contexts.', 'chosen': 'Client retention and loyalty are critical for the success of any business, especially for small companies operating in a competitive landscape. To strategically enhance client retention and loyalty, a small company owner can follow the action plan outlined below, considerin

In [2]:
# --- 3. Загрузка модели и токенайзера ---
# Для Reward Model нам нужна модель с одной головой для регрессии (num_labels=1)
model = AutoModelForSequenceClassification.from_pretrained(
    SFT_MODEL_ID,
    num_labels=1,
    torch_dtype=torch.float32,
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained(SFT_MODEL_ID)

# Установка pad_token, если он отсутствует
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    model.config.pad_token_id = model.config.eos_token_id


Some weights of LlamaForSequenceClassification were not initialized from the model checkpoint at HuggingFaceTB/SmolLM2-135M-Instruct and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [3]:
# --- 4. Препроцессинг данных для RewardTrainer ---
# RewardTrainer ожидает, что в датасете будут колонки 'chosen' и 'rejected'
# Наш датасет уже имеет эти колонки, но они содержат полный диалог.
# Нам нужно отформатировать их в виде "промпт + ответ".
def preprocess_function(examples):
    new_examples = {
        "chosen": [],
        "rejected": [],
    }
    for prompt, chosen, rejected in zip(examples["prompt"], examples["chosen"], examples["rejected"]):
        # Формируем полный текст для chosen и rejected
        # Формат: <|user|>\n{prompt}<|end|>\n<|assistant|>\n{response}<|end|>
        # Этот формат ожидает модель SmolLM2-Instruct
        chosen_text = f"<|user|>\n{prompt}<|end|>\n<|assistant|>\n{chosen}<|end|>"
        rejected_text = f"<|user|>\n{prompt}<|end|>\n<|assistant|>\n{rejected}<|end|>"
        new_examples["chosen"].append(chosen_text)
        new_examples["rejected"].append(rejected_text)

    return new_examples

train_dataset = train_dataset.map(preprocess_function, batched=True, num_proc=4)
eval_dataset = eval_dataset.map(preprocess_function, batched=True, num_proc=4)

In [4]:
# --- 5. Обучение Reward Model ---
training_args = RewardConfig(
    output_dir="./rm_trainer_logs",
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=1,
    learning_rate=5e-5,
    fp16=True, # Включаем смешанную точность
    logging_steps=100,
    eval_steps=500,
    save_steps=500,
    eval_strategy="steps",
    save_total_limit=2,
    remove_unused_columns=False, # Важно для RewardTrainer
    report_to="none", # Отключаем логирование в wandb/tensorboard для простоты
    max_length=MAX_LENGTH,
)

trainer = RewardTrainer(
    model=model,
    processing_class=tokenizer,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
)

print("\nНачинаем обучение Reward Model...")
trainer.train()
print("Обучение Reward Model завершено.")

Detected kernel version 5.4.270, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


[2025-06-23 05:22:55,565] [INFO] [real_accelerator.py:239:get_accelerator] Setting ds_accelerator to cuda (auto detect)


/usr/bin/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status
/usr/bin/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status
You're using a GPT2TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.



Начинаем обучение Reward Model...


Step,Training Loss,Validation Loss,Accuracy
500,0.674100,0.634278,0.644444
1000,0.662500,0.624383,0.661581
1500,0.634800,0.606437,0.662968


...

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┓
┃ chosen_text                                   ┃ rejected_text                                ┃ logits           ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━┩
│ <|user|>                                      │ <|user|>                                     │ [0.692, 0.308]   │
│ If an AI system has the ability to correct    │ If an AI system has the ability to correct   │                  │
│ its own code and continuously update its      │ its own code and continuously update its     │                  │
│ model, would it be considered as a negative   │ model, would it be considered as a negative  │                  │
│ entropy process?                              │ entropy process?                             │                  │
│ <extra_id_1>Assistant                         │ <extra_id_1>Assistant                        │                  │
│ Yes, an AI system with this level of          │ Yes, an AI system with this level of         │                  │
│ self-improvement would certainly be           │ self-improvement would certainly be          │                  │
│ considered a negative entropy process, as it  │ considered a negative entropy process, as it │                  │
│ would generate order and information from     │ would generate order and information from    │                  │
│ disorder and randomness. This type of system  │ disorder and randomness. This type of system │                  │
│ would be highly organized and sophisticated,  │ would be highly organized and sophisticated, │                  │
│ and would continually increase its own        │ and would continually increase its own       │                  │
│ complexity and intelligence. In this sense,   │ complexity and intelligence. In this sense,  │                  │
│ it would be a source of "life", in a similar  │ it would be a source of "life", in a similar │                  │
│ way as living organisms are sources of        │ way as living organisms are sources of       │                  │
│ negative entropy.                             │ negative entropy.                            │                  │
│ <extra_id_1>User                              │ <extra_id_1>User                             │                  │
│ Someone once said the true source of life is  │ Someone once said the true source of life is │                  │
│ free energy, or Gibbs free energy             │ free energy, or Gibbs free energy            │                  │
│ <extra_id_1>Assistant                         │ <extra_id_1>Assistant                        │                  │
│ Yes, this statement seems to be an accurate   │ Yes, this statement seems to be an accurate  │                  │
│ description of the fundamental nature of      │ description of the fundamental nature of     │                  │
│ living organisms. Life depends on the ability │ living organisms. Life depends on the        │                  │
│ of organisms to generate energy and maintain  │ ability of organisms to generate energy and  │                  │
│ homeostasis through complex metabolic         │ maintain homeostasis through complex         │                  │
│ processes, and therefore requires the         │ metabolic processes, and therefore requires  │                  │
│ presence of free energy. This concept is      │ the presence of free energy. This concept is │                  │
│ fundamental to understanding the nature of    │ fundamental to understanding the nature of   │                  │
│ living systems.                               │ living systems.                              │                  │
│ <extra_id_1>User                              │ <extra_id_1>User                             │                  │
│ So if an AI system has the ability to correct │ So if 

/trinity/home/a.anokhin/.local/lib/python3.11/site-packages/trl/trainer/utils.py:790: UserWarning: There are 1 out of 722 instances where the predictions for both options are equal. These instances are ignored in the accuracy computation.
  warnings.warn(


┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┓
┃ chosen_text                                   ┃ rejected_text                                ┃ logits           ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━┩
│ <|user|>                                      │ <|user|>                                     │ [0.4393, 0.5607] │
│ If an AI system has the ability to correct    │ If an AI system has the ability to correct   │                  │
│ its own code and continuously update its      │ its own code and continuously update its     │                  │
│ model, would it be considered as a negative   │ model, would it be considered as a negative  │                  │
│ entropy process?                              │ entropy process?                             │                  │
│ <extra_id_1>Assistant                         │ <extra_id_1>Assistant                        │                  │
│ Yes, an AI system with this level of          │ Yes, an AI system with this level of         │                  │
│ self-improvement would certainly be           │ self-improvement would certainly be          │                  │
│ considered a negative entropy process, as it  │ considered a negative entropy process, as it │                  │
│ would generate order and information from     │ would generate order and information from    │                  │
│ disorder and randomness. This type of system  │ disorder and randomness. This type of system │                  │
│ would be highly organized and sophisticated,  │ would be highly organized and sophisticated, │                  │
│ and would continually increase its own        │ and would continually increase its own       │                  │
│ complexity and intelligence. In this sense,   │ complexity and intelligence. In this sense,  │                  │
│ it would be a source of "life", in a similar  │ it would be a source of "life", in a similar │                  │
│ way as living organisms are sources of        │ way as living organisms are sources of       │                  │
│ negative entropy.                             │ negative entropy.                            │                  │
│ <extra_id_1>User                              │ <extra_id_1>User                             │                  │
│ Someone once said the true source of life is  │ Someone once said the true source of life is │                  │
│ free energy, or Gibbs free energy             │ free energy, or Gibbs free energy            │                  │
│ <extra_id_1>Assistant                         │ <extra_id_1>Assistant                        │                  │
│ Yes, this statement seems to be an accurate   │ Yes, this statement seems to be an accurate  │                  │
│ description of the fundamental nature of      │ description of the fundamental nature of     │                  │
│ living organisms. Life depends on the ability │ living organisms. Life depends on the        │                  │
│ of organisms to generate energy and maintain  │ ability of organisms to generate energy and  │                  │
│ homeostasis through complex metabolic         │ maintain homeostasis through complex         │                  │
│ processes, and therefore requires the         │ metabolic processes, and therefore requires  │                  │
│ presence of free energy. This concept is      │ the presence of free energy. This concept is │                  │
│ fundamental to understanding the nature of    │ fundamental to understanding the nature of   │                  │
│ living systems.                               │ living systems.                              │                  │
│ <extra_id_1>User                              │ <extra_id_1>User                             │                  │
│ So if an AI system has the ability to correct │ So if 

/trinity/home/a.anokhin/.local/lib/python3.11/site-packages/trl/trainer/utils.py:790: UserWarning: There are 1 out of 722 instances where the predictions for both options are equal. These instances are ignored in the accuracy computation.
  warnings.warn(


Обучение Reward Model завершено.


In [3]:
import torch
import torch.nn.functional as F
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModelForSequenceClassification
from tqdm import tqdm
import numpy as np
import os

# --- 1. Конфигурация ---
SFT_MODEL_ID = "HuggingFaceTB/SmolLM2-135M-Instruct"
# Путь к вашей обученной Reward Model (убедитесь, что он правильный)
RM_PATH = "/trinity/home/a.anokhin/test_rl/test_task/rm_trainer_logs/checkpoint-1623" 
REINFORCE_SAVE_PATH = "./reinforce_model_smollm2_notebook"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Гиперпараметры для REINFORCE
LEARNING_RATE = 1e-6
NUM_EPOCHS = 1 
BATCH_SIZE = 4      # Размер батча для генерации и обучения
MAX_PROMPTS = 2000  # Ограничим количество промптов для ускорения обучения
MAX_NEW_TOKENS = 128 # Максимальная длина генерируемого ответа
EVAL_BATCH_SIZE = 8
EVAL_PROMPTS = 200  # Количество промптов для валидации

print(f"Используемое устройство: {DEVICE}")

# --- 2. Вспомогательный класс для Baseline ---
class MovingAverageBaseline:
    """
    Простой класс для подсчета скользящего среднего (moving average)
    награды, которое будет использоваться в качестве baseline.
    """
    def __init__(self, momentum=0.9):
        self.momentum = momentum
        self.value = 0
        self.is_initialized = False

    def update(self, new_values):
        """Обновляет baseline на основе нового батча наград."""
        if not self.is_initialized:
            self.value = new_values.mean().item()
            self.is_initialized = True
        else:
            new_mean = new_values.mean().item()
            self.value = self.momentum * self.value + (1 - self.momentum) * new_mean
    
    def get(self):
        """Возвращает текущее значение baseline."""
        return self.value

# --- 3. Загрузка моделей и токенайзера ---
print("Загрузка моделей и токенайзера...")

# Policy Model (модель, которую мы будем обучать)
policy_model = AutoModelForCausalLM.from_pretrained(SFT_MODEL_ID).to(DEVICE)

# Reward Model (модель для оценки, не обучается)
reward_model = AutoModelForSequenceClassification.from_pretrained(RM_PATH).to(DEVICE)
reward_model.eval() # Переводим в режим оценки

# Токенайзер
tokenizer = AutoTokenizer.from_pretrained(SFT_MODEL_ID, padding_side='left')
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    policy_model.config.pad_token_id = policy_model.config.eos_token_id

# --- 4. Загрузка и подготовка данных ---
print("Загрузка и подготовка датасета...")
# Мы будем использовать validation split, чтобы не подглядывать в train данные
# и иметь отложенную выборку для финальной оценки
dataset = load_dataset("juyoungml/HelpSteer2-binarized", split="train")
dataset = dataset.train_test_split(test_size=0.1, seed=42)

# Разделяем на train (для REINFORCE) и validation (для итоговой оценки)
train_prompts = dataset["train"].select(range(MAX_PROMPTS))
val_prompts = dataset["test"].select(range(EVAL_PROMPTS))

def format_prompt(prompt_text):
    """Форматирует промпт для модели SmolLM2"""
    return f"<|user|>\n{prompt_text}<|end|>\n<|assistant|>\n"

# --- 5. Функция оценки ---
@torch.no_grad()
def evaluate_model(model, tokenizer, prompts, batch_size):
    """
    Функция для оценки средней награды модели на наборе промптов.
    """
    model.eval()
    all_rewards = []
    
    print(f"Оценка модели на {len(prompts)} промптах...")
    for i in tqdm(range(0, len(prompts), batch_size)):
        batch_prompts_text = prompts[i:i+batch_size]['prompt']
        formatted_prompts = [format_prompt(p) for p in batch_prompts_text]

        inputs = tokenizer(formatted_prompts, return_tensors="pt", padding=True).to(DEVICE)
        
        # Генерация ответов
        generated_outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            pad_token_id=tokenizer.pad_token_id,
            do_sample=True, # Важно для разнообразия
            top_k=50,
            top_p=0.95
        )
        
        # Декодируем сгенерированные ответы
        generated_texts = tokenizer.batch_decode(generated_outputs, skip_special_tokens=True)
        
        # Оцениваем с помощью Reward Model
        reward_inputs = tokenizer(
            generated_texts, 
            return_tensors="pt", 
            padding=True, 
            truncation=True, 
            max_length=2048
        ).to(DEVICE)
        
        rewards = reward_model(**reward_inputs).logits.squeeze(-1)
        all_rewards.extend(rewards.cpu().tolist())
        
    return np.mean(all_rewards)

# --- 6. Основной цикл обучения REINFORCE ---

# Оптимизатор для policy model
optimizer = torch.optim.Adam(policy_model.parameters(), lr=LEARNING_RATE)
baseline = MovingAverageBaseline(momentum=0.99)

# Оцениваем исходную SFT модель
sft_avg_reward = evaluate_model(policy_model, tokenizer, val_prompts, EVAL_BATCH_SIZE)
print(f"\nСредняя награда исходной SFT модели: {sft_avg_reward:.4f}")

policy_model.train() # Переводим policy model в режим обучения

print("\nНачинаем обучение с помощью REINFORCE w/ baseline...")
for epoch in range(NUM_EPOCHS):
    print(f"--- Эпоха {epoch + 1} / {NUM_EPOCHS} ---")
    
    # Перемешиваем данные на каждой эпохе
    shuffled_prompts = train_prompts.shuffle(seed=42+epoch)
    
    for i in tqdm(range(0, len(shuffled_prompts), BATCH_SIZE)):
        # 1. Формируем батч промптов
        batch_prompts_text = shuffled_prompts[i:i+BATCH_SIZE]['prompt']
        formatted_prompts = [format_prompt(p) for p in batch_prompts_text]
        prompt_inputs = tokenizer(formatted_prompts, return_tensors="pt", padding=True).to(DEVICE)
        prompt_len = prompt_inputs.attention_mask.size(1)

        # 2. Сэмплируем действия (генерируем ответы) из текущей политики
        # output_scores=True - нужно для подсчета логарифма вероятности
        generated_outputs = policy_model.generate(
            **prompt_inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            pad_token_id=tokenizer.pad_token_id,
            do_sample=True,
            top_k=50,
            top_p=0.95,
            return_dict_in_generate=True,
            output_scores=True
        )
        
        # 3. Получаем награду (Reward) от Reward Model
        full_sequences_ids = generated_outputs.sequences
        full_sequences_text = tokenizer.batch_decode(full_sequences_ids, skip_special_tokens=True)
        
        reward_inputs = tokenizer(
            full_sequences_text, 
            return_tensors="pt", 
            padding=True, 
            truncation=True, 
            max_length=2048
        ).to(DEVICE)
        
        with torch.no_grad():
            rewards = reward_model(**reward_inputs).logits.squeeze(-1)

        # 4. Обновляем и получаем baseline
        baseline.update(rewards)
        b = baseline.get()
        
        # 5. Считаем Advantage
        advantages = rewards - b
        
        # 6. Считаем log(pi(a|s)) - логарифм вероятности сгенерированной последовательности
        
        # Получаем сгенерированную часть
        generated_ids = full_sequences_ids[:, prompt_len:]
        
        # Прогоняем полную последовательность (промпт + ответ) через модель, чтобы получить logits
        full_logits = policy_model(full_sequences_ids).logits

        # Нас интересуют только logits для сгенерированных токенов
        generated_logits = full_logits[:, prompt_len-1:-1, :]
        
        # Считаем log-вероятности для каждого сгенерированного токена
        log_probs = F.cross_entropy(
            generated_logits.reshape(-1, generated_logits.size(-1)), 
            generated_ids.reshape(-1), 
            reduction='none'
        )
        log_probs = log_probs.reshape(generated_ids.size())
        
        # Суммируем log-вероятности по всей сгенерированной последовательности для каждого примера в батче
        # Мы должны учитывать только реальные сгенерированные токены, а не паддинг
        mask = (generated_ids != tokenizer.pad_token_id).float()
        log_probs_sum = (log_probs * mask).sum(dim=1)
        
        # 7. Считаем loss для REINFORCE
        # loss = -E[advantage * log(pi(a|s))]
        # Мы минимизируем -loss, поэтому убираем минус
        loss = (advantages * log_probs_sum).mean()
        
        if i % 50 == 0:
            print(f"\nШаг {i}/{len(shuffled_prompts)//BATCH_SIZE} | "
                  f"Loss: {loss.item():.4f} | "
                  f"Mean Reward: {rewards.mean().item():.4f} | "
                  f"Baseline: {b:.4f}")
        
        # 8. Backpropagation и шаг оптимизатора
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
# --- 7. Финальная оценка и сохранение модели ---
print("\nОбучение завершено. Финальная оценка модели...")
reinforce_avg_reward = evaluate_model(policy_model, tokenizer, val_prompts, EVAL_BATCH_SIZE)
print(f"\nСредняя награда исходной SFT модели: {sft_avg_reward:.4f}")
print(f"Средняя награда REINFORCE модели: {reinforce_avg_reward:.4f}")

if reinforce_avg_reward > sft_avg_reward:
    print("\n✅ Средняя награда на отложенной выборке выросла!")
else:
    print("\n⚠️ Средняя награда на отложенной выборке не выросла или уменьшилась.")

# Сохраняем модель и токенайзер
print(f"Сохранение REINFORCE модели в {REINFORCE_SAVE_PATH}...")
policy_model.save_pretrained(REINFORCE_SAVE_PATH)
tokenizer.save_pretrained(REINFORCE_SAVE_PATH)
print("Модель успешно сохранена.")

Используемое устройство: cuda
Загрузка моделей и токенайзера...
Загрузка и подготовка датасета...
Оценка модели на 200 промптах...


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 25/25 [01:23<00:00,  3.33s/it]



Средняя награда исходной SFT модели: 1.7637

Начинаем обучение с помощью REINFORCE w/ baseline...
--- Эпоха 1 / 1 ---


  0%|▏                                                                                                                      | 1/500 [00:03<27:23,  3.29s/it]


Шаг 0/500 | Loss: 10.6357 | Mean Reward: 1.4655 | Baseline: 1.4655


...


Шаг 1900/500 | Loss: -294.9619 | Mean Reward: 1.1947 | Baseline: 1.8515


100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 500/500 [27:43<00:00,  3.33s/it]



Обучение завершено. Финальная оценка модели...
Оценка модели на 200 промптах...


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 25/25 [01:23<00:00,  3.35s/it]



Средняя награда исходной SFT модели: 1.7637
Средняя награда REINFORCE модели: 1.9335

✅ Средняя награда на отложенной выборке выросла!
Сохранение REINFORCE модели в ./reinforce_model_smollm2_notebook...
Модель успешно сохранена.


In [1]:
import torch
import torch.nn.functional as F
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModelForSequenceClassification
from tqdm import tqdm
import numpy as np
import os

# --- 1. Конфигурация ---
SFT_MODEL_ID = "HuggingFaceTB/SmolLM2-135M-Instruct"
# ## ИЗМЕНЕНИЕ ##
# Укажите путь к вашей новой RM, обученной на регрессии
RM_PATH = "/trinity/home/a.anokhin/test_rl/test_task/rm_regression_trainer_logs/checkpoint-3251" 
REINFORCE_SAVE_PATH = "./reinforce_model_smollm2_regression_-loss"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Гиперпараметры для REINFORCE
LEARNING_RATE = 1e-6
NUM_EPOCHS = 1 
BATCH_SIZE = 4      
MAX_PROMPTS = 2000  
MAX_NEW_TOKENS = 128 
EVAL_BATCH_SIZE = 8
EVAL_PROMPTS = 200

print(f"Используемое устройство: {DEVICE}")

# --- 2. Вспомогательный класс для Baseline (остается без изменений) ---
class MovingAverageBaseline:
    # ... (код класса остается прежним)
    def __init__(self, momentum=0.9):
        self.momentum = momentum
        self.value = 0
        self.is_initialized = False

    def update(self, new_values):
        if not self.is_initialized:
            self.value = new_values.mean().item()
            self.is_initialized = True
        else:
            new_mean = new_values.mean().item()
            self.value = self.momentum * self.value + (1 - self.momentum) * new_mean
    
    def get(self):
        return self.value

# --- 3. Загрузка моделей и токенайзера ---
print("Загрузка моделей и токенайзера...")

policy_model = AutoModelForCausalLM.from_pretrained(SFT_MODEL_ID).to(DEVICE)

# ## ИЗМЕНЕНИЕ ##
# Загружаем модель с указанием `problem_type` для корректной инициализации.
# Хотя это не строго обязательно для инференса, это хорошая практика.
reward_model = AutoModelForSequenceClassification.from_pretrained(
    RM_PATH,
    problem_type="regression", 
    num_labels=1
).to(DEVICE)
reward_model.eval() 

tokenizer = AutoTokenizer.from_pretrained(SFT_MODEL_ID, padding_side='left')
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    policy_model.config.pad_token_id = policy_model.config.eos_token_id

# --- 4. Загрузка и подготовка данных (остается без изменений) ---
print("Загрузка и подготовка датасета...")
dataset = load_dataset("juyoungml/HelpSteer2-binarized", split="train")
dataset = dataset.train_test_split(test_size=0.1, seed=42)
train_prompts = dataset["train"].select(range(MAX_PROMPTS))
val_prompts = dataset["test"].select(range(EVAL_PROMPTS))

def format_prompt(prompt_text):
    return f"<|user|>\n{prompt_text}<|end|>\n<|assistant|>\n"

# --- 5. Функция оценки ---
@torch.no_grad()
def evaluate_model(model, tokenizer, prompts, batch_size):
    model.eval()
    all_rewards = []
    
    print(f"Оценка модели на {len(prompts)} промптах...")
    for i in tqdm(range(0, len(prompts), batch_size)):
        batch_prompts_text = prompts[i:i+batch_size]['prompt']
        formatted_prompts = [format_prompt(p) for p in batch_prompts_text]
        inputs = tokenizer(formatted_prompts, return_tensors="pt", padding=True).to(DEVICE)
        
        generated_outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            pad_token_id=tokenizer.pad_token_id,
            do_sample=True,
            top_k=50,
            top_p=0.95
        )
        
        generated_texts = tokenizer.batch_decode(generated_outputs, skip_special_tokens=True)
        
        reward_inputs = tokenizer(
            generated_texts, 
            return_tensors="pt", 
            padding=True, 
            truncation=True, 
            max_length=2048 # Оставляем запас по длине для RM
        ).to(DEVICE)
        
        # ## ИЗМЕНЕНИЕ ##
        # Выход регрессионной модели уже имеет размер [batch_size, 1].
        # Мы просто убираем последнюю размерность.
        rewards = reward_model(**reward_inputs).logits.squeeze(dim=-1)
        all_rewards.extend(rewards.cpu().tolist())
        
    return np.mean(all_rewards)

# --- 6. Основной цикл обучения REINFORCE ---
optimizer = torch.optim.Adam(policy_model.parameters(), lr=LEARNING_RATE)
baseline = MovingAverageBaseline(momentum=0.99)

sft_avg_reward = evaluate_model(policy_model, tokenizer, val_prompts, EVAL_BATCH_SIZE)
print(f"\nСредняя награда исходной SFT модели: {sft_avg_reward:.4f}")

policy_model.train() 

print("\nНачинаем обучение с помощью REINFORCE w/ baseline (с регрессионной RM)...")
for epoch in range(NUM_EPOCHS):
    print(f"--- Эпоха {epoch + 1} / {NUM_EPOCHS} ---")
    
    shuffled_prompts = train_prompts.shuffle(seed=42+epoch)
    
    for i in tqdm(range(0, len(shuffled_prompts), BATCH_SIZE)):
        # 1. Формируем батч (без изменений)
        batch_prompts_text = shuffled_prompts[i:i+BATCH_SIZE]['prompt']
        formatted_prompts = [format_prompt(p) for p in batch_prompts_text]
        prompt_inputs = tokenizer(formatted_prompts, return_tensors="pt", padding=True).to(DEVICE)
        prompt_len = prompt_inputs.attention_mask.size(1)

        # 2. Сэмплируем действия (без изменений)
        generated_outputs = policy_model.generate(
            **prompt_inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            pad_token_id=tokenizer.pad_token_id,
            do_sample=True,
            top_k=50,
            top_p=0.95,
            return_dict_in_generate=True,
            output_scores=True
        )
        
        # 3. Получаем награду (Reward) от Reward Model
        full_sequences_ids = generated_outputs.sequences
        full_sequences_text = tokenizer.batch_decode(full_sequences_ids, skip_special_tokens=True)
        
        reward_inputs = tokenizer(
            full_sequences_text, 
            return_tensors="pt", 
            padding=True, 
            truncation=True, 
            max_length=2048
        ).to(DEVICE)
        
        with torch.no_grad():
            # ## ИЗМЕНЕНИЕ ##
            # То же самое, что и в функции evaluate_model.
            # Выход модели [batch_size, 1] -> squeeze(dim=-1) -> [batch_size]
            rewards = reward_model(**reward_inputs).logits.squeeze(dim=-1)

        # 4. Обновляем baseline (без изменений)
        baseline.update(rewards)
        b = baseline.get()
        
        # 5. Считаем Advantage (без изменений)
        advantages = rewards - b
        
        # 6. Считаем log(pi(a|s)) (без изменений)
        generated_ids = full_sequences_ids[:, prompt_len:]
        full_logits = policy_model(full_sequences_ids).logits
        generated_logits = full_logits[:, prompt_len-1:-1, :]
        
        # ## ИЗМЕНЕНИЕ (микро-оптимизация) ##
        # Вместо cross_entropy с reduction='none' можно посчитать log_softmax и gather,
        # что может быть чуть более явно, но ваш способ абсолютно корректен.
        # Оставим ваш способ, так как он работает и понятен.
        log_probs_per_token = F.cross_entropy(
            generated_logits.reshape(-1, generated_logits.size(-1)), 
            generated_ids.reshape(-1), 
            reduction='none'
        )
        log_probs_per_token = log_probs_per_token.reshape(generated_ids.size())
        
        mask = (generated_ids != tokenizer.pad_token_id).float()
        log_probs_sum = (log_probs_per_token * mask).sum(dim=1)
        
        # 7. Считаем loss для REINFORCE
        # ## ИЗМЕНЕНИЕ ##
        # loss = -E[advantage * log(pi(a|s))]
        # Мы минимизируем эту величину. Ваша формула `(advantages * log_probs_sum).mean()`
        # максимизирует Advantage, если log_probs > 0, что неверно.
        # Правильный policy gradient loss должен иметь знак минус.
        loss = -(advantages.detach() * log_probs_sum).mean()
        
        if i % 50 == 0:
            print(f"\nШаг {i}/{len(shuffled_prompts)//BATCH_SIZE} | "
                  f"Loss: {loss.item():.4f} | "
                  f"Mean Reward: {rewards.mean().item():.4f} | "
                  f"Baseline: {b:.4f}")
        
        # 8. Backpropagation (без изменений)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
# --- 7. Финальная оценка и сохранение модели (без изменений) ---
print("\nОбучение завершено. Финальная оценка модели...")
reinforce_avg_reward = evaluate_model(policy_model, tokenizer, val_prompts, EVAL_BATCH_SIZE)
print(f"\nСредняя награда исходной SFT модели: {sft_avg_reward:.4f}")
print(f"Средняя награда REINFORCE модели: {reinforce_avg_reward:.4f}")

if reinforce_avg_reward > sft_avg_reward:
    print("\n✅ Средняя награда на отложенной выборке выросла!")
else:
    print("\n⚠️ Средняя награда на отложенной выборке не выросла или уменьшилась.")

print(f"Сохранение REINFORCE модели в {REINFORCE_SAVE_PATH}...")
policy_model.save_pretrained(REINFORCE_SAVE_PATH)
tokenizer.save_pretrained(REINFORCE_SAVE_PATH)
print("Модель успешно сохранена.")

Используемое устройство: cuda
Загрузка моделей и токенайзера...
Загрузка и подготовка датасета...
Оценка модели на 200 промптах...


100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 25/25 [01:30<00:00,  3.61s/it]



Средняя награда исходной SFT модели: 3.1314

Начинаем обучение с помощью REINFORCE w/ baseline (с регрессионной RM)...
--- Эпоха 1 / 1 ---


  0%|▏                                                                                                                | 1/500 [00:03<29:30,  3.55s/it]


Шаг 0/500 | Loss: -119.5793 | Mean Reward: 2.4542 | Baseline: 2.4542


...


Обучение завершено. Финальная оценка модели...
Оценка модели на 200 промптах...


100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 25/25 [01:27<00:00,  3.51s/it]


Средняя награда исходной SFT модели: 3.1314
Средняя награда REINFORCE модели: 3.1658

✅ Средняя награда на отложенной выборке выросла!
Сохранение REINFORCE модели в ./reinforce_model_smollm2_regression_-loss...
[2025-06-23 06:32:31,409] [INFO] [real_accelerator.py:239:get_accelerator] Setting ds_accelerator to cuda (auto detect)



/usr/bin/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status
/usr/bin/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status


Модель успешно сохранена.
